# Zhang & Wamsley
## Data Prep
### Records
We dropped the patients whom have not experienced any dreaming in this experiment. Therefore not relevant for our project

### PSG
Created new folder just containing the EEGs relevant for our study. Being the analysis of the subjects whom experienced dreaming

### Reports
Created a new dataset containing the dream descriptions of our relevant subjects. 

### Final dataset
Now we will perform a merge that will contain the EEGs associated with the dream description attached. 

In [ ]:
import pandas as pd

# Define the file path
file_path = '/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Records.csv'

# Read the CSV file
records_df = pd.read_csv(file_path)

# Display the first few rows of the dataframe
records_df.head()

In [ ]:
filtered_df = records_df[records_df['Experience'] == 2]
filtered_df

In [ ]:
# Correct the output path by splitting the file_path and appending the new file name
output_path = file_path.rsplit('/', 1)[0] + '/Reports_with_exp.csv'

# Save the filtered DataFrame to the new CSV file
filtered_df.to_csv(output_path, index=False)

In [ ]:
import os
import shutil
import pandas as pd

# Paths
filtered_csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Records_with_exp.csv"
eeg_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG/"
backup_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered/"

# Step 1: Create backup folder if it doesn't exist
os.makedirs(backup_folder, exist_ok=True)

# Step 2: Load the filtered CSV and extract valid filenames
df = pd.read_csv(filtered_csv_path)
valid_filenames = set(df['Filename'].astype(str))  # Convert to string & store in set

# Step 3: Copy matching EEG files to the new folder
for eeg_file in os.listdir(eeg_folder):
    if eeg_file in valid_filenames:
        src_path = os.path.join(eeg_folder, eeg_file)
        dest_path = os.path.join(backup_folder, eeg_file)
        shutil.copy2(src_path, dest_path)  # Copies with metadata
        print(f"✅ Copied: {src_path} → {dest_path}")

print("🎉 Done! All relevant EEG files are now in the 'PSG_Filtered' folder.")


In [ ]:
# REPORTS FILE PATH
reports_path = '/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/DATA/Reports.csv'

# Read the CSV file
reports_df = pd.read_csv(reports_path)

# Display the first few rows of the dataframe
reports_df.info()

In [ ]:
import pandas as pd

# Paths
filtered_csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Records_with_exp.csv"
reports_csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports.csv"
filtered_reports_csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered.csv"

# Step 1: Load both CSVs
df_filtered = pd.read_csv(filtered_csv_path)  # CSV with the filenames we need
 # df_reports = pd.read_csv(reports_csv_path)  # Original reports CSV

# Step 2: Keep only rows where 'Filename' matches
valid_filenames = set(df_filtered['Filename'].astype(str))  # Convert to string & store in set
reports_df_filtered = reports_df[reports_df['Filename'].astype(str).isin(valid_filenames)]

# Step 3: Save the new filtered reports CSV
reports_df_filtered.to_csv(filtered_reports_csv_path, index=False)

print(f"🎉 Done! Filtered reports saved to {filtered_reports_csv_path}")


In [ ]:
filtered_reports_csv_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Reports_Filtered.csv"
# Read the filtered reports CSV
filtered_reports_df = pd.read_csv(filtered_reports_csv_path)

filtered_reports_df.info()

# Next Steps
## EEG Preprocessing Pipeline
Convert EEG signals into a format suitable for training a transformer-based model.

## Dream description Preprocessing Pipeline
Convert textual descriptions into a structured form usable for text-to-image models.

In [ ]:
import mne
import os
from glob import glob

# Define the path to the EEG dataset folder
eeg_folder_path = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered/"

# Get all .edf files in the folder
eeg_files = sorted(glob(os.path.join(eeg_folder_path, "*.edf")))

# Dictionary to store raw EEG data
eeg_data = {}

# Load each EEG file
for file in eeg_files:
    subject_id = os.path.basename(file).replace(".edf", "")  # Extract filename without extension
    print(f"Loading {subject_id} ...")
    
    # Load EEG
    raw = mne.io.read_raw_edf(file, preload=True, verbose=False)
    
    # Store it in dictionary
    eeg_data[subject_id] = raw

    # Print metadata
    print(raw.info)
    print("\n" + "-"*50 + "\n")

# Print number of loaded EEGs
print(f"✅ Successfully loaded {len(eeg_data)} EEG files!")


### Process EEGs and store them in new folder

In [ ]:
import mne
import os
import glob

# Path to the filtered EEG files
eeg_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/PSG_Filtered/"
filtered_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Processed_EEGs/"

# Ensure the output folder exists
os.makedirs(filtered_folder, exist_ok=True)

# Get all .edf files
eeg_files = glob.glob(os.path.join(eeg_folder, "*.edf"))

for eeg_file in eeg_files:
    # Load EEG file
    raw = mne.io.read_raw_edf(eeg_file, preload=True)
    
    # Apply band-pass filter (keeping 5-45 Hz)
    raw.filter(l_freq=5, h_freq=45, fir_design='firwin')

    # Save the filtered EEG file as .fif
    output_path = os.path.join(filtered_folder, os.path.basename(eeg_file).replace(".edf", ".fif"))
    raw.save(output_path, overwrite=True)

    print(f"✅ Filtered and saved: {output_path}")

print("\n🎯 EEG Filtering Completed! All files saved in:", filtered_folder)


# Visualize Filtered Signals

In [ ]:
import mne
import os
import glob
import matplotlib.pyplot as plt

# Path to the filtered EEG files
filtered_folder = "/Users/mayer/Desktop/IE - BDBA3rd Year/Designing AI/IndProject/DATA/Zhang & Wamsley 2019/Data/Processed_EEGs/"

# Get all .fif files in the folder
filtered_files = glob.glob(os.path.join(filtered_folder, "*.fif"))

# Select a few files for visualization
sample_files = filtered_files[:5]  # For example, visualize the first 5 files

# Set the duration (in seconds) for the visualization
duration = 70  # 20 seconds to view the signal

# Plot the EEG signals for the selected files
for eeg_file in sample_files:
    # Load the filtered EEG file
    raw = mne.io.read_raw_fif(eeg_file, preload=True)
    
    # Select the first 20 seconds of the data for visualization
    raw.plot(duration=duration, n_channels=10, scalings='auto', title=os.path.basename(eeg_file))

    # Optionally, show specific channels
    # raw.plot(duration=duration, picks=["Fp1", "Fp2", "Fz"], title=os.path.basename(eeg_file))

    print(f"Visualizing: {eeg_file}")
    
plt.show()


# REVIEW EEGs AND WHICH 'VARS' TAKE INTO ACCOUNT AND FOR HOW LONG